---

## 🚖 NYC TAXI ZONE LOOKUP EXTRACTION & LOAD

> _Seamlessly extract, stage, and transform the Taxi Zone Lookup reference for analytical excellence._

---

### 🗂️ Extract CSV  
Download [Taxi Zone Lookup](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv) to secure landing zone in ADLS.

---

### 💾 Read & Transform  
Load and enrich zone dimension—casting types and stamping effective dates.

---

### 🛡️ Load as Bronze Table  
Stage conformant data in:  
**`NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP`**

---

In [0]:
import urllib.request
import os
import shutil
from datetime import datetime

In [0]:
# =====================================================
# CALCULATE START TIME
# =====================================================

load_start_time = datetime.now()

#### EXTRACT `TAXI_ZONE_LOOKUP.csv` FROM API TO ADLS

In [0]:
url = f'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
response = urllib.request.urlopen(url)

dir_path = f'/Volumes/nyctaxi/landing/yellow_taxi/lookup/'
os.makedirs(dir_path, exist_ok = True)

local_file_path = '/Volumes/nyctaxi/landing/yellow_taxi/lookup/taxi_zone_lookup.csv'
with open(local_file_path, 'wb') as f:
    shutil.copyfileobj(response, f)

#### READ FROM VOLUME

In [0]:
taxi_lookup_df = (spark.read.format('csv')
                            .option('header', 'true')
                            .load('/Volumes/nyctaxi/landing/yellow_taxi/lookup/taxi_zone_lookup.csv')
                            )
print(f'Records Effeted: {taxi_lookup_df.count()}')

In [0]:
from pyspark.sql.functions import col, cast, current_timestamp, lit
from pyspark.sql.types import IntegerType, TimestampType

taxi_lookup_trans_df = taxi_lookup_df.select(
                                            col('LocationID').cast(IntegerType()).alias('location_id'),
                                            col('Borough').alias('borough'),
                                            col('Zone').alias('zone'),
                                            col('service_zone'),
                                            current_timestamp().alias('effective_start_date'),
                                            lit(None).cast(TimestampType()).alias('effective_end_date')
                                            )

taxi_lookup_trans_df.limit(100).display()

#### LOAD LOOKUP INTO
- `NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP`

In [0]:
taxi_lookup_trans_df.write.mode('overwrite').saveAsTable('NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP')

#### OBSERVABILITY METRICES

In [0]:
#############################
### PERFORM AUDIT LOGGING ###
#############################

from datetime import datetime

from pyspark.sql.functions import (
    col,
    lit,
    regexp_extract,
    round
)

from pyspark.sql.types import (
    DecimalType,
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    TimestampType
)

# =====================================================
# CAPTURE START TIME
# =====================================================

load_start_time = datetime.now()

# =====================================================
# WORKFLOW PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("event_type", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("layer", "")
dbutils.widgets.text("status", "")
dbutils.widgets.text("message", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

# =====================================================
# RETRIEVE PARAMETERS
# =====================================================

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
layer = dbutils.widgets.get("layer")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT DATE
# =====================================================

try:
    if not raw_event_time or raw_event_time.startswith("{{"):
        event_time = datetime.now().date()
    else:
        event_time = datetime.fromisoformat(raw_event_time).date()
except:
    event_time = datetime.now().date()

# =====================================================
# RUNTIME METRICS
# =====================================================

user_name = spark.sql(
    "SELECT current_user()"
).first()[0]

# =====================================================
# FILE METADATA EXTRACTION
# =====================================================

file_metadata_df = (
    taxi_lookup_df
    .select(
        col("_metadata.file_name").alias("file_name"),
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_size").cast("bigint").alias("file_size_bytes"),
        col("_metadata.file_modification_time").alias("file_modified_time")
    )
    .distinct()
)

# =====================================================
# DERIVE ADDITIONAL FILE ATTRIBUTES
# =====================================================

file_metadata_df = (
    file_metadata_df
    .withColumn(
        "file_extension",
        regexp_extract(
            col("file_name"),
            r"\.([^\.]+)$",
            1
        )
    )
    .withColumn(
        "source_system",
        lit("NYCTAXI")
    )
    .withColumn(
        "source_folder",
        regexp_extract(
            col("file_path"),
            r".*/([^/]+)/[^/]+$",
            1
        )
    )
    .withColumn(
        "file_size_mb",
        round(
            col("file_size_bytes") / (1024 * 1024),
            2
        ).cast(DecimalType(18, 2))
    )
    .withColumn(
        "file_created_time",
        col("file_modified_time")
    )
)

# =====================================================
# MAIN LOAD LOGIC
# =====================================================

try:

    record_count = taxi_lookup_df.count()

    status = "SUCCESS"
    event_type = "LOAD_SUCCESS"
    message = f"Loaded {record_count} records into {target_table}"

except Exception as e:

    record_count = 0
    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)

# =====================================================
# CAPTURE END TIME
# =====================================================

load_end_time = datetime.now()

# =====================================================
# BUILD AUDIT DATAFRAME
# =====================================================

audit_df = (
    file_metadata_df
    .withColumn("log_id", lit(log_id))
    .withColumn("run_id", lit(run_id))
    .withColumn("event_time", lit(event_time).cast("date"))
    .withColumn("event_type", lit(event_type))
    .withColumn("source_table", lit(source_table))
    .withColumn("target_table", lit(target_table))
    .withColumn("layer", lit(layer))
    .withColumn("record_count", lit(record_count).cast("bigint"))
    .withColumn("status", lit(status))
    .withColumn("message", lit(message))
    .withColumn("user_name", lit(user_name))
    .withColumn("notebook_path", lit(notebook_path))
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("load_start_time", lit(load_start_time).cast("timestamp"))
    .withColumn("load_end_time", lit(load_end_time).cast("timestamp"))
)

# =====================================================
# COLUMN ORDER
# =====================================================

audit_df = audit_df.select(
    "log_id",
    "run_id",
    "event_time",
    "event_type",
    "source_table",
    "target_table",
    "layer",
    "record_count",
    "status",
    "message",
    "user_name",
    "notebook_path",
    "pipeline_name",
    "load_start_time",
    "load_end_time",
    "file_name",
    "file_path",
    "file_extension",
    "source_system",
    "source_folder",
    "file_size_bytes",
    "file_size_mb",
    "file_created_time",
    "file_modified_time"
)

# =====================================================
# WRITE TO AUDIT TABLE
# =====================================================

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG"
    )

print(
    f"Audit logging completed successfully for Run ID: {run_id}"
)

In [0]:
%skip
%sql
-- VALIDATE THE RECORDS
SELECT 
TIMESTAMPDIFF(
           SECOND,
           load_start_time,
           load_end_time
       ) / 60.0 AS load_duration_minutes,
       *
FROM NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG;

In [0]:
dbutils.notebook.exit('TAXI LOOKUP FILE HAS BEEN LOADED INTO NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP')